# 03 · Anatomy of a Front-Page Post

What separates a 500-point story from one that gets 3 points and dies? We analyse title features, timing, and metadata to find the real predictors.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.loader import db
from src.nlp import title_features, extract_domain
from src.viz import set_style, heatmap_2d, bar_chart, save

set_style()
con = db()

In [2]:
stories = con.execute("""
    SELECT
        title,
        url,
        score,
        comment_count,
        posted_at,
        HOUR(posted_at)    AS hour_utc,
        DAYOFWEEK(posted_at) AS dow,   -- 0=Sunday in DuckDB
        YEAR(posted_at)    AS year
    FROM stories
    WHERE score IS NOT NULL
      AND year BETWEEN 2015 AND 2024
""").df()

stories = title_features(stories)
stories['domain'] = stories['url'].apply(extract_domain)
print(f'{len(stories):,} stories')

## When to post: score by hour × day-of-week

In [3]:
pivot = stories.groupby(['dow', 'hour_utc'])['score'].median().unstack()
pivot.index = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']

fig = heatmap_2d(
    pivot,
    title='Median story score by day-of-week and hour (UTC)',
    xlabel='Hour (UTC)',
    ylabel='Day of week',
)
save(fig, '../data/fig_timing_heatmap.png')
plt.show()

best_slot = pivot.stack().idxmax()
print(f'Best posting slot: {best_slot[0]} at {best_slot[1]:02d}:00 UTC')
print('(Note: based on 2015+ data to exclude early-HN era skew)')

## Title length vs score

In [4]:
bins = pd.cut(stories['title_len_words'], bins=[0,5,8,11,14,17,30], labels=['1–5','6–8','9–11','12–14','15–17','18+'])
length_score = stories.groupby(bins, observed=True)['score'].median().reset_index()
length_score.columns = ['title_words', 'median_score']
display(length_score)

fig = bar_chart(
    list(length_score['title_words'].astype(str)),
    list(length_score['median_score']),
    title='Median score by title word count',
    xlabel='Median score',
    horizontal=True,
)
save(fig, '../data/fig_title_length.png')
plt.show()

## Questions vs statements vs numbers

In [5]:
feature_scores = {
    'Question (ends with ?)': stories[stories['title_is_question']]['score'].median(),
    'Contains a number': stories[stories['title_has_number']]['score'].median(),
    'Positive sentiment': stories[stories['title_sentiment'] > 0.1]['score'].median(),
    'Negative sentiment': stories[stories['title_sentiment'] < -0.1]['score'].median(),
    'Neutral sentiment': stories[stories['title_sentiment'].between(-0.1, 0.1)]['score'].median(),
    'All stories (baseline)': stories['score'].median(),
}

fig = bar_chart(
    list(feature_scores.keys()),
    list(feature_scores.values()),
    title='Median score by title feature',
    xlabel='Median score',
)
save(fig, '../data/fig_title_features.png')
plt.show()

print('\nMedian score by title feature:')
print(pd.Series(feature_scores).sort_values(ascending=False).to_string())


## Discussion maximizers: high comments, low score

In [6]:
controversy = stories[(stories['score'] >= 2) & stories['comment_count'].notna()].copy()
controversy['controversy_ratio'] = controversy['comment_count'] / (controversy['score'] + 1)

print('Top 20 discussion-maximizing posts (most comments relative to score):')
top_controversial = controversy.nlargest(20, 'controversy_ratio')[['title', 'year', 'score', 'comment_count', 'controversy_ratio']]
display(top_controversial)